# RealSaS — Full Mage Runtime-v4 admission + native smoke

Fail-closed product smoke. Exact FIT2/P1Q/foreground/assembly/camera inputs are resolved by SHA-256. Render runs only after full-Mage render-free admission certificates pass. No model training, no completion, no 600–700 MB product-graph load.


In [ ]:
# 1) Mount Drive + pin exact code
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, pathlib
REPO = pathlib.Path('/content/RealSaS-OPT')
CODE_COMMIT = '8f4a14cf0d77e94935d85399901c1a4c3bb2729e'
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/merynz/RealSaS-OPT.git',str(REPO)], check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin','playback-stack-v1-20260916'], check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--force',CODE_COMMIT], check=True)
head = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip()
assert head == CODE_COMMIT, (head, CODE_COMMIT)
os.chdir(REPO)
print('CODE_PIN_PASS', head)


In [ ]:
# 2) Dependencies + exact input resolution (no filename guessing for mechanics/cameras)
import json, subprocess, pathlib, os
subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','cmake','g++','libarchive-dev','libpng-dev','zlib1g-dev'], check=True)
subprocess.run(['python','-m','pip','install','-q','numpy','scipy','pillow','scikit-image'], check=True)

SEARCH_ROOT = pathlib.Path('/content/drive/MyDrive')
WORK = pathlib.Path('/content/drive/MyDrive/RealSaS_RuntimeV4_Smoke_20260918')
WORK.mkdir(parents=True, exist_ok=True)
RESOLUTION = WORK / 'MAGE_RUNTIME_V4_EXACT_INPUT_RESOLUTION_V1.json'
subprocess.run([
    'python','experiments/playback_stack_v1/resolve_mage_full_assembly_runtime_v4_inputs_v1.py',
    '--search-root',str(SEARCH_ROOT),
    '--output',str(RESOLUTION),
], check=True)
inputs = json.loads(RESOLUTION.read_text())
assert inputs['status'] == 'PASS__EXACT_MAGE_RUNTIME_V4_INPUTS_RESOLVED'
print(json.dumps({k:inputs[k] for k in ['p1q_dir','foreground_dir','assembly_dir','fit2_surface','skeleton','fit2_skin','cameras']}, indent=2))


In [ ]:
# 3) Build only the native Runtime-v4 consumer
import subprocess, pathlib
BUILD = pathlib.Path('/content/realsas-runtime-v4-build')
subprocess.run(['cmake','-S','runtime/realsas_cpp','-B',str(BUILD),'-DCMAKE_BUILD_TYPE=Release','-DREALSAS_BUILD_TESTS=OFF'], check=True)
subprocess.run(['cmake','--build',str(BUILD),'--parallel','2'], check=True)
RUNTIME_DEMO = BUILD / 'realsas_runtime_v4_demo'
assert RUNTIME_DEMO.is_file(), RUNTIME_DEMO
print('NATIVE_RUNTIME_BUILD_PASS', RUNTIME_DEMO)


In [ ]:
# 4) Full Mage admission -> package -> native idle/run render -> 8-view GIFs
# Admission is inside the runner and hard-fails before packaging/render if any certificate fails.
import subprocess, pathlib, json
OUT = WORK / 'FULL_MAGE_RUNTIME_V4_SMOKE'
CACHE = WORK / 'runtime_v4_cache'
cmd = [
    'python','experiments/playback_stack_v1/run_mage_full_assembly_runtime_v4_native_smoke_v1.py',
    '--cameras', *inputs['cameras'],
    '--p1q-dir', inputs['p1q_dir'],
    '--foreground-dir', inputs['foreground_dir'],
    '--assembly-dir', inputs['assembly_dir'],
    '--fit2-surface', inputs['fit2_surface'],
    '--skeleton', inputs['skeleton'],
    '--fit2-skin', inputs['fit2_skin'],
    '--expected-p1q-manifest', inputs['expected_hashes']['p1q_manifest'],
    '--expected-foreground-manifest', inputs['expected_hashes']['foreground_manifest'],
    '--expected-assembly-manifest', inputs['expected_hashes']['assembly_manifest'],
    '--output-dir', str(OUT),
    '--cache-root', str(CACHE),
    '--runtime-demo', str(RUNTIME_DEMO),
    '--render-sample-count','17',
    '--contact-thumb','256',
]
proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(proc.stdout)
if proc.returncode != 0:
    raise RuntimeError('FULL_MAGE_RUNTIME_V4_SMOKE_FAIL')
result_path = OUT / 'MAGE_FULL_ASSEMBLY_RUNTIME_V4_NATIVE_SMOKE_V1.json'
result = json.loads(result_path.read_text())
assert result['status'] == 'PASS__FULL_MAGE_RUNTIME_V4_NATIVE_RENDER'
assert result['completion_used'] is False and result['new_pixels_generated'] is False
print(json.dumps({
    'status': result['status'],
    'package_MB': result['package_megabytes_decimal'],
    'runtime_binary_MB': result['runtime_binary_megabytes_decimal'],
    'cache_hit': result['cache_hit'],
    'archive_sha256': result['package']['archive_sha256'],
    'visual_outputs': result['visual_outputs'],
}, indent=2))


In [ ]:
# 5) Show final 8-view idle/run results
from IPython.display import display, Image
for clip_id in ('mage_fit1_idle_v2','mage_fit1_run_v2'):
    row = result['visual_outputs'][clip_id]
    print(clip_id, row['gif_sha256'])
    display(Image(filename=row['gif']))
